In [14]:
from pathlib import Path
import json
import re
import hcl2
import difflib
import yaml
import webbrowser
import logging
import networkx as nx
from pyvis.network import Network
from zss import Node, simple_distance
from itertools import combinations
from clone_detection_parallel import detect_clones_zss as detect_clones_zss_parallel
from clone_detection_smart import detect_clones_smart
from terraform_ast import build_terraform_ast


In [15]:
parsed_asts = {}

def find_iac_files(root, limit=None):
    """Trova tutti i file IaC nella directory data."""
    exts = ('.tf',)
    files = []
    for p in Path(root).rglob('*'):
        try:
            if p.is_file() and p.suffix in exts:
                files.append(p)
                if limit and len(files) >= limit:
                    break
        except (PermissionError, OSError) as e:
            # Skip files/directories we don't have permission to access
            logging.debug(f"Skipping {p}: {e}")
            continue
    return files

def parse_file(path):
    global parsed_asts
    if path in parsed_asts:
        return parsed_asts[path]
    
    try:
        with open(path, 'r', encoding='utf-8') as f:
            if path.suffix == '.tf':
                data = hcl2.load(f)
            elif path.suffix in ('.yaml', '.yml'):
                data = yaml.safe_load(f)
            elif path.suffix == '.json':
                data = json.load(f)
        
        parsed_asts[path] = data
        return data
    except Exception as e:
        print(f"Failed to parse {path}: {e}")
        return None

def to_zss_tree(node, label='root'):
    """
    Recursively convert a dictionary/list-based AST into a zss.Node tree.
    """
    if isinstance(node, dict):
        zss_node = Node(label)
        for k, v in sorted(node.items()): # Sort keys for consistency
            zss_node.addkid(to_zss_tree(v, label=k))
        return zss_node
    elif isinstance(node, list):
        zss_node = Node(label)
        for item in node:
            # Use a generic label for list items
            zss_node.addkid(to_zss_tree(item, label='item'))
        return zss_node
    else:
        # For literals, you can use their type or a constant value
        return Node(str(type(node).__name__))

def detect_clones_zss(files, threshold=5):
    """
    Detects clones using tree edit distance with zss.
    This version uses the canonical AST from terraform_ast.py.
    Returns a list of clone pairs and their distance.
    """
    asts = {}
    for path in files:
        if path.suffix == '.tf':
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    content = f.read()
                # Use the new structured AST builder for .tf files
                asts[path] = build_terraform_ast(content)
            except Exception as e:
                print(f"Failed to build AST for {path}: {e}")
        # Note: Non-Terraform files will be skipped by this function.

    clone_pairs = []
    # Compare every file with every other file
    for path1, path2 in combinations(asts.keys(), 2):
        distance = simple_distance(asts[path1], asts[path2])

        if distance <= threshold:
            print(f"Found potential clone pair: ({path1.name}, {path2.name}) with distance {distance}")
            clone_pairs.append((path1, path2, distance))

    return clone_pairs

def report_zss(clone_groups):
    if not clone_groups:
        print("No clones found.")
        return
    
    for i, group in enumerate(clone_groups):
        print(f"\nClone group {i+1}: {len(group)} files")
        for p in sorted(list(group)): # Sort for consistent output
            print(f"  - {p}")


In [16]:
def visualize_clones_diff(clone_groups):
    """
    Generates a text-based diff for each pair of files in a clone group.
    """
    if not clone_groups:
        print("No clones to visualize.")
        return

    print(f"\n--- Visualizing {len(clone_groups)} Clone Groups ---")

    for i, group in enumerate(clone_groups):
        # Iterate over every pair of files in the clone group
        for path1, path2 in combinations(sorted(list(group)), 2):
            print(f"\n{'='*80}")
            print(f"Clone Group {i+1}: DIFF between '{path1.name}' and '{path2.name}'")
            print(f"{'='*80}")

            try:
                with open(path1, 'r', encoding='utf-8') as f1, open(path2, 'r', encoding='utf-8') as f2:
                    file1_lines = f1.readlines()
                    file2_lines = f2.readlines()

                # Generate and print the diff
                diff = difflib.unified_diff(
                    file1_lines,
                    file2_lines,
                    fromfile=str(path1),
                    tofile=str(path2),
                )
                for line in diff:
                    print(line, end='')

            except Exception as e:
                print(f"Could not generate diff for {path1.name} and {path2.name}: {e}")

def _find_ast_diff(d1, d2, path=""):
    """Helper to find differing values in two ASTs (dicts)."""
    diffs = set()
    # Check keys in d1 against d2
    for k in d1:
        p = f"{path}.{k}" if path else k
        if k not in d2:
            diffs.add(f"Key '{p}' only in first file.")
            continue
        
        v1, v2 = d1[k], d2[k]
        if isinstance(v1, dict) and isinstance(v2, dict):
            diffs.update(_find_ast_diff(v1, v2, path=p))
        elif isinstance(v1, list) and isinstance(v2, list):
            if len(v1) != len(v2):
                diffs.add(f"List length differs at '{p}' ({len(v1)} vs {len(v2)})")
            else:
                # Iterate through list items
                for i, (item1, item2) in enumerate(zip(v1, v2)):
                    item_path = f"{p}[{i}]"
                    if isinstance(item1, dict) and isinstance(item2, dict):
                        diffs.update(_find_ast_diff(item1, item2, path=item_path))
                    elif item1 != item2:
                        diffs.add(f"Value differs at '{item_path}': ('{item1}' vs '{item2}')")
        elif v1 != v2:
            diffs.add(f"Value differs at '{p}': ('{v1}' vs '{v2}')")

    # Check for keys in d2 that are not in d1
    for k in d2:
        p = f"{path}.{k}" if path else k
        if k not in d1:
            diffs.add(f"Key '{p}' only in second file.")
            
    return diffs


def _is_variable_candidate(value):
    """Determines if a value is a primitive that can be turned into a variable."""
    return isinstance(value, (str, int, float, bool))

def _infer_type(value):
    """Infers the Terraform type string."""
    if isinstance(value, bool): return "bool"
    if isinstance(value, int): return "number"
    if isinstance(value, float): return "number"
    if isinstance(value, list): return "list(any)"
    return "string"

def _sanitize_var_name(path):
    """Converts an AST path (resource.aws_instance.web.ami) into a variable name (ami)."""
    # Heuristic: try to use the last segment. If generic (e.g. 'name', 'tags'), prepend parent.
    parts = path.split('.')
    name = parts[-1]
    if name in ['name', 'tags', 'type', 'id', 'enabled'] and len(parts) > 1:
        return f"{parts[-2]}_{name}"
    return name

def _hcl_value(val):
    """Formats a Python value into an HCL string."""
    if isinstance(val, bool):
        return str(val).lower()
    if isinstance(val, (int, float)):
        return str(val)
    if isinstance(val, str):
        # Handle HEREDOC or multiline if necessary, strict quoting for now
        return f'"{val}"'
    if isinstance(val, list):
        items = [_hcl_value(x) for x in val]
        return f"[{', '.join(items)}]"
    return f'"{val}"' # Fallback

def _identify_param_differences(ast1, ast2, current_path=""):
    """
    Compare two ASTs structure-wise. 
    Returns a dict: { 'path.to.param': {'val1': v1, 'val2': v2, 'type': 'string'} }
    """
    diffs = {}
    
    # If types differ completely, we can't parameterize easily
    if type(ast1) != type(ast2):
        return {}

    if isinstance(ast1, dict):
        all_keys = set(ast1.keys()) | set(ast2.keys())
        for k in all_keys:
            new_path = f"{current_path}.{k}" if current_path else k
            if k not in ast1 or k not in ast2:
                # Structural difference (block missing). 
                # Complex to handle in simple refactoring, ignoring for now.
                continue 
            diffs.update(_identify_param_differences(ast1[k], ast2[k], new_path))
            
    elif isinstance(ast1, list):
        # Naive list comparison: assumed fixed order for clones
        for i, (item1, item2) in enumerate(zip(ast1, ast2)):
            new_path = f"{current_path}[{i}]"
            diffs.update(_identify_param_differences(item1, item2, new_path))
            
    else:
        # Leaf node (Primitive value)
        if ast1 != ast2 and _is_variable_candidate(ast1):
            diffs[current_path] = {
                'val1': ast1,
                'val2': ast2,
                'type': _infer_type(ast1)
            }
            
    return diffs

def _render_hcl_recursive(node, variable_map, current_path="", indent_level=0):
    """
    Rebuilds HCL from AST, injecting 'var.xyz' where differences were found.
    FIX: Now correctly includes list indices [i] in paths for root-level resources.
    """
    indent = "  " * indent_level
    lines = []

    # Check if this specific leaf node is a known variable
    if current_path in variable_map:
        var_name = variable_map[current_path]
        return f"var.{var_name}"

    if isinstance(node, dict):
        for k, v in node.items():
            new_path = f"{current_path}.{k}" if current_path else k
            
            # Special handling for resource/module/data blocks to make them look like HCL
            if indent_level == 0 and k in ['resource', 'data', 'module']:
                if isinstance(v, list):
                    # FIX: Enumerate to track the index [i] matching the diff map
                    for i, item in enumerate(v):
                        for type_name, resources in item.items():
                            for res_name, props in resources.items():
                                lines.append(f'\n{k} "{type_name}" "{res_name}" {{')
                                # FIX: Include [i] in the path
                                sub_path = f"{new_path}[{i}].{type_name}.{res_name}" 
                                lines.append(_render_hcl_recursive(props, variable_map, sub_path, indent_level+1))
                                lines.append("}\n")
                continue

            # Standard attribute or child block
            if isinstance(v, dict):
                lines.append(f"{indent}{k} {{")
                lines.append(_render_hcl_recursive(v, variable_map, new_path, indent_level+1))
                lines.append(f"{indent}}}")
            elif isinstance(v, list):
                 # Heuristic: if list of dicts, it's likely repeated blocks (like ingress {})
                 if len(v) > 0 and isinstance(v[0], dict):
                     for i, item in enumerate(v):
                         lines.append(f"{indent}{k} {{")
                         lines.append(_render_hcl_recursive(item, variable_map, f"{new_path}[{i}]", indent_level+1))
                         lines.append(f"{indent}}}")
                 else:
                     val_str = _render_hcl_recursive(v, variable_map, new_path, indent_level)
                     lines.append(f"{indent}{k} = {val_str}")
            else:
                # Leaf node value
                val_str = _render_hcl_recursive(v, variable_map, new_path, indent_level)
                lines.append(f"{indent}{k} = {val_str}")

    elif isinstance(node, list):
        # Literal list (e.g. security_groups = [...])
        rendered_items = [_render_hcl_recursive(x, variable_map, f"{current_path}[{i}]", indent_level) for i, x in enumerate(node)]
        return f"[{', '.join(rendered_items)}]"
    
    else:
        # Literal value
        return _hcl_value(node)

    return "\n".join(lines)

def _generate_smart_module_tf(ast1, diff_map):
    """
    Generates variables.tf and main.tf using the AST and Diff Map.
    """
    # 1. Prepare Variable Map (Path -> Variable Name)
    variable_map = {} # path -> var_name
    var_definitions = [] # Lines for variables.tf
    
    for path, details in diff_map.items():
        var_name = _sanitize_var_name(path)
        # Ensure unique variable names
        counter = 1
        original_name = var_name
        while var_name in variable_map.values():
            var_name = f"{original_name}_{counter}"
            counter += 1
            
        variable_map[path] = var_name
        
        # Build variable definition
        var_def = f'variable "{var_name}" {{\n'
        var_def += f'  description = "Refactored from {path}"\n'
        var_def += f'  type        = {details["type"]}\n'
        var_def += '}\n'
        var_definitions.append(var_def)

    # 2. Render main.tf (The common code)
    # We use AST1 as the "Skeleton" and inject vars into it
    main_tf_content = _render_hcl_recursive(ast1, variable_map)

    return "\n".join(var_definitions), main_tf_content, variable_map

def _generate_smart_module_call(module_name, diff_map, variable_map, specific_ast_values):
    """
    Generates the module usage block.
    specific_ast_values: 'val1' (left file) or 'val2' (right file) to pick specific values.
    """
    call_lines = [f'module "{module_name}" {{', f'  source = "./modules/{module_name}"']
    
    for path, var_name in variable_map.items():
        # Get the specific value for this file instance
        if specific_ast_values == 'left':
            val = diff_map[path]['val1']
        else:
            val = diff_map[path]['val2']
            
        call_lines.append(f'  {var_name} = {_hcl_value(val)}')

    call_lines.append('}')
    return "\n".join(call_lines)

def _generate_module_tf(base_ast, differences):
    """Generates the HCL for a new Terraform module."""
    variables = {}
    # Naively create variable names from the diff path
    for diff in differences:
        # Example diff: "Value differs at '.resource.aws_instance.web.instance_type': ('t2.micro' vs 't2.large')"
        try:
            path_part = diff.split("'")[1]
            # A simple heuristic for variable names
            var_name = path_part.split('.')[-1]
            variables[var_name] = {'path': path_part}
        except IndexError:
            continue

    # --- Generate variables.tf content ---
    var_tf_lines = ['# variables.tf for the new module\n']
    for name in sorted(variables.keys()):
        var_tf_lines.append(f'variable "{name}" {{')
        var_tf_lines.append('  description = "Autogenerated variable"')
        var_tf_lines.append('}\n')

    # --- Generate main.tf content (very simplified) ---
    # This is a complex task. For this example, we'll just show the concept
    # by replacing values in a string representation of the original resource.
    # A real implementation would need a robust HCL generator.
    main_tf_lines = ['# main.tf for the new module\n']
    
    # Handle that 'resource' can be a list of dicts or a single dict
    resource_block = base_ast.get('resource')
    if isinstance(resource_block, list):
        resource_block = resource_block[0] # Use the first resource block as a template
    
    # Let's find the first resource block to use as a template
    resource_key = next((k for k in resource_block), None) if resource_block else None
    if resource_key:
        resource_name = next(iter(resource_block[resource_key]), None)
        if resource_name:
            # Crude representation of the resource
            main_tf_lines.append(f'resource "{resource_key}" "{resource_name}" {{')
            
            # The resource block can be a list with one dict or just the dict
            resource_attributes = resource_block[resource_key][resource_name]
            if isinstance(resource_attributes, list):
                resource_attributes = resource_attributes[0]

            for key, value in resource_attributes.items():
                 # Check if this attribute is a variable
                is_variable = False
                for var_name, var_info in variables.items():
                    if var_info['path'].endswith(f'.{key}'):
                        main_tf_lines.append(f'  {key} = var.{var_name}')
                        is_variable = True
                        break
                if not is_variable:
                    main_tf_lines.append(f'  {key} = "{value}"') # Note: simplistic quoting
            main_tf_lines.append('}')

    return '\n'.join(var_tf_lines), '\n'.join(main_tf_lines)

def _generate_module_call(module_name, differences, original_values):
    """Generates the HCL for calling the new module."""
    call_lines = [f'module "{module_name}" {{', '  source = "./modules/{module_name}"\n']
    
    variables = {}
    for diff in differences:
        try:
            path_part = diff.split("'")[1]
            var_name = path_part.split('.')[-1]
            # Extract the first value as an example
            value = diff.split("'")[3]
            variables[var_name] = value
        except IndexError:
            continue

    for name, value in sorted(variables.items()):
        call_lines.append(f'  {name} = "{value}"')

    call_lines.append('}')
    return '\n'.join(call_lines)


In [17]:
def suggest_refactorings(clone_groups):
    """Analyzes clone groups and suggests refactoring into modules."""
    if not clone_groups:
        return

    print(f"\n{'='*80}")
    print("Refactoring Suggestions (Smart Mode - FIXED)")
    print(f"{'='*80}")

    for i, group in enumerate(clone_groups):
        if len(group) < 2:
            continue

        paths = sorted(list(group))
        path1, path2 = paths[0], paths[1]
        
        print(f"\n--- Clone Group {i+1} ---")
        print(f"Prototype Files: {path1.name} <--> {path2.name}")
        
        ast1 = parse_file(path1)
        ast2 = parse_file(path2)

        if ast1 and ast2:
            diff_map = _identify_param_differences(ast1, ast2)
            
            if not diff_map:
                print("  Status: Identical files (deduplication candidate).")
                continue
                
            print(f"  Detected {len(diff_map)} parametric differences.")
            
            # Generate Module Code
            var_tf, main_tf, var_map = _generate_smart_module_tf(ast1, diff_map)
            
            print("\n  [Generated: variables.tf]")
            print("-" * 30)
            print(var_tf)
            
            print("\n  [Generated: main.tf (Abstracted Logic)]")
            print("-" * 30)
            print(main_tf)

            # Generate Calls
            print(f"\n  [Refactoring: {path1.name}]")
            print("-" * 30)
            print(_generate_smart_module_call(f"common_module_{i+1}", diff_map, var_map, 'left'))
            
            print(f"\n  [Refactoring: {path2.name}]")
            print("-" * 30)
            print(_generate_smart_module_call(f"common_module_{i+1}", diff_map, var_map, 'right'))

In [18]:
import networkx as nx
from pyvis.network import Network

def _generate_clone_graph_html(clone_pairs, graph_filename="clone_graph.html"):
    """
    Generates an interactive clone graph visualization using NetworkX and Pyvis.
    Returns the path to the generated HTML file.
    """
    if not clone_pairs:
        return None

    # 1. Build NetworkX Graph
    G = nx.Graph()
    
    # clone_pairs is expected to be a list of tuples: [(path1, path2, distance), ...]
    for path1, path2, distance in clone_pairs:
        p1_name = path1.name
        p2_name = path2.name
        
        # Add nodes (files)
        G.add_node(p1_name, title=str(path1), group=1, size=15)
        G.add_node(p2_name, title=str(path2), group=1, size=15)
            
        # Add edge. Distance is inverted for value (smaller distance = stronger link/thicker edge)
        # We cap the distance value to make it visible
        edge_weight = max(1, 100 - distance*10) 
        G.add_edge(p1_name, p2_name, value=edge_weight, title=f"Distance: {distance:.2f}")

    # 2. Convert to Pyvis Network
    nt = Network(height="600px", width="100%", bgcolor="#222222", font_color="white", notebook=False)
    nt.from_nx(G)
    
    # 3. Save to HTML file
    nt.save_graph(graph_filename)
    
    return graph_filename

def generate_comprehensive_report(clone_pairs, clone_groups, output_filename="clone_report.html"):
    """
    Generates a comprehensive HTML report with all visualizations:
    - Overview statistics
    - Interactive clone graph (embedded)
    - Side-by-side diffs
    - Refactoring suggestions
    """
    if not clone_pairs and not clone_groups:
        print("No clones to report.")
        return

    html_parts = [
        '<!DOCTYPE html><html><head><meta charset="UTF-8">',
        '<title>Clone Detection Report</title>',
        '<style>',
        '* { margin: 0; padding: 0; box-sizing: border-box; }',
        'body { font-family: "Segoe UI", Tahoma, Geneva, Verdana, sans-serif; background: #f5f5f5; color: #333; }',
        'header { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 2rem; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }',
        'header h1 { font-size: 2.5rem; margin-bottom: 0.5rem; }',
        'header p { font-size: 1.1rem; opacity: 0.9; }',
        '.container { max-width: 1400px; margin: 0 auto; padding: 2rem; }',
        '.stats { display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 1rem; margin-bottom: 2rem; }',
        '.stat-card { background: white; padding: 1.5rem; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); text-align: center; }',
        '.stat-card h3 { color: #667eea; font-size: 2rem; margin-bottom: 0.5rem; }',
        '.stat-card p { color: #666; font-size: 0.9rem; }',
        'nav { background: white; padding: 1rem; border-radius: 8px; margin-bottom: 2rem; box-shadow: 0 2px 8px rgba(0,0,0,0.1); position: sticky; top: 20px; z-index: 100; }',
        'nav a { color: #667eea; text-decoration: none; padding: 0.5rem 1rem; margin: 0 0.25rem; border-radius: 4px; transition: background 0.3s; display: inline-block; }',
        'nav a:hover { background: #f0f0f0; }',
        '.section { background: white; padding: 2rem; margin-bottom: 2rem; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); }',
        '.section h2 { color: #667eea; margin-bottom: 1.5rem; padding-bottom: 0.5rem; border-bottom: 2px solid #667eea; }',
        '.clone-group { background: #f9f9f9; padding: 1.5rem; margin-bottom: 1.5rem; border-radius: 8px; border-left: 4px solid #667eea; }',
        '.clone-group h3 { color: #764ba2; margin-bottom: 1rem; }',
        '.file-list { background: white; padding: 1rem; border-radius: 4px; margin-bottom: 1rem; }',
        '.file-list li { padding: 0.5rem; border-bottom: 1px solid #eee; list-style-position: inside; }',
        '.file-list li:last-child { border-bottom: none; }',
        'table.diff { font-family: "Courier New", monospace; font-size: 0.85rem; width: 100%; border: 1px solid #ddd; margin: 1rem 0; }',
        'table.diff td { padding: 0.25rem 0.5rem; white-space: pre-wrap; word-wrap: break-word; }',
        '.diff_header { background-color: #667eea !important; color: white !important; font-weight: bold; }',
        '.diff_next { background-color: #e0e0e0; }',
        '.diff_add { background-color: #d4edda; }',
        '.diff_chg { background-color: #fff3cd; }',
        '.diff_sub { background-color: #f8d7da; }',
        'pre { background: #f4f4f4; padding: 1rem; border-radius: 4px; overflow-x: auto; font-size: 0.9rem; }',
        'code { background: #f4f4f4; padding: 0.2rem 0.4rem; border-radius: 3px; font-family: "Courier New", monospace; }',
        '.refactoring { background: #e7f3ff; padding: 1rem; border-radius: 4px; margin: 1rem 0; border-left: 4px solid #2196F3; }',
        '.graph-container { width: 100%; height: 600px; border: 1px solid #ddd; border-radius: 4px; margin: 1rem 0; }',
        'footer { text-align: center; padding: 2rem; color: #666; font-size: 0.9rem; }',
        '</style>',
        '</head><body>',
        '<header>',
        '<div class="container">',
        '<h1> Clone Detection Report</h1>',
        '</div>',
        '</header>',
        '<div class="container">'
    ]

    # Statistics section
    total_files = len(set([p for pair in clone_pairs for p in [pair[0], pair[1]]]))
    total_groups = len(clone_groups)
    total_pairs = len(clone_pairs)
    
    html_parts.extend([
        '<div class="stats">',
        '<div class="stat-card"><h3>{}</h3><p>Clone Groups</p></div>'.format(total_groups),
        '<div class="stat-card"><h3>{}</h3><p>Clone Pairs</p></div>'.format(total_pairs),
        '<div class="stat-card"><h3>{}</h3><p>Files Involved</p></div>'.format(total_files),
        '</div>'
    ])

    # Navigation
    html_parts.extend([
        '<nav>',
        '<a href="#graph"> Graph View</a>',
        '<a href="#diffs"> Code Diffs</a>',
        '<a href="#refactoring"> Refactoring</a>',
        '</nav>'
    ])

    # Generate the Pyvis network for embedding
    html_parts.append('<section id="graph" class="section"><h2>Clone Graph Visualization</h2>')
    
    graph_filename = _generate_clone_graph_html(clone_pairs)
    
    if graph_filename:
        html_parts.append(f"""
        <p>Click on nodes to view file paths. Edge thickness represents similarity (thicker = more similar).</p>
        <iframe src="{graph_filename}" width="100%" height="600" style="border: none;"></iframe>
        <p>Alternatively, view the graph in a separate tab: <a href="{graph_filename}" target="_blank">Open Graph Visualization ({graph_filename})</a></p>
        """)
    else:
        html_parts.append("<p>Not enough clone pairs found to generate a meaningful graph visualization.</p>")
        
    html_parts.append('</section>')

    # Code Diffs Section
    html_parts.extend([
        '<div class="section" id="diffs">',
        '<h2> Side-by-Side Code Comparisons</h2>'
    ])

    html_diff = difflib.HtmlDiff(tabsize=4, wrapcolumn=50)
    for i, group in enumerate(clone_groups):
        html_parts.append(f'<div class="clone-group">')
        html_parts.append(f'<h3>Clone Group {i+1}</h3>')
        html_parts.append('<ul class="file-list">')
        for path in sorted(list(group)):
            html_parts.append(f'<li>{path}</li>')
        html_parts.append('</ul>')
        
        for path1, path2 in combinations(sorted(list(group)), 2):
            try:
                with open(path1, 'r', encoding='utf-8') as f1, open(path2, 'r', encoding='utf-8') as f2:
                    file1_lines = f1.readlines()
                    file2_lines = f2.readlines()
                
                diff_table = html_diff.make_table(
                    file1_lines,
                    file2_lines,
                    fromdesc=str(path1.name),
                    todesc=str(path2.name),
                    context=True,
                    numlines=2
                )
                html_parts.append(diff_table)
            except Exception as e:
                html_parts.append(f'<p> Could not generate diff: {e}</p>')
        
        html_parts.append('</div>')
    
    html_parts.append('</div>')

# 4. Refactoring Suggestions (UPDATED SECTION)
    html_parts.append('<section id="refactoring" class="section"><h2>4. Refactoring Suggestions (Candidates for Module Extraction)</h2>')
    
    if clone_groups:
        for i, group in enumerate(clone_groups):
            if len(group) < 2:
                continue

            paths = sorted(list(group))
            path1, path2 = paths[0], paths[1]
            module_name = f"common_module_{i+1}"

            try:
                ast1 = parse_file(path1)
                ast2 = parse_file(path2)
            except Exception as e:
                html_parts.append(f'<div class="refactoring-block"><h4>Error analyzing group {i+1}</h4><p>Failed to parse AST for file pair: {e}</p></div>')
                continue

            if ast1 and ast2:
                # *** NEW SMART LOGIC INTEGRATION ***
                diff_map = _identify_param_differences(ast1, ast2)
                
                if not diff_map:
                    html_parts.append(f'<div class="refactoring-block"><h4>Refactoring Suggestion for Group {i+1}</h4><p>Files {path1.name} and {path2.name} appear structurally identical. Recommend simple file deduplication.</p></div>')
                    continue

                # 1. Generate Module Files
                var_tf, main_tf, var_map = _generate_smart_module_tf(ast1, diff_map)
                
                # 2. Generate Module Calls
                call_1 = _generate_smart_module_call(module_name, diff_map, var_map, 'left')
                call_2 = _generate_smart_module_call(module_name, diff_map, var_map, 'right')
                
                # 3. Format into HTML
                html_block = f"""
                <div class="refactoring-block">
                    <h4>Refactoring Suggestion for Group {i+1} ({len(group)} files)</h4>
                    <p><strong>Prototype Files:</strong> <code>{path1.name}</code> and <code>{path2.name}</code>. Detected <strong>{len(diff_map)}</strong> parameters that differ.</p>
                    
                    <h5>&#x1F4BE; Proposed New Module Structure (<code>./modules/{module_name}</code>)</h5>
                    <div class="code-container">
                        <div class="code-section">
                            <h5>variables.tf</h5>
                            <div class="code-block"><pre>{var_tf}</pre></div>
                        </div>
                        <div class="code-section">
                            <h5>main.tf (Abstracted Logic)</h5>
                            <div class="code-block"><pre>{main_tf}</pre></div>
                        </div>
                    </div>

                    <h5>&#x1F504; Replacement Code</h5>
                    <div class="code-container">
                        <div class="code-section">
                            <h5>Replaces original code in <code>{path1.name}</code></h5>
                            <div class="code-block"><pre>{call_1}</pre></div>
                        </div>
                        <div class="code-section">
                            <h5>Replaces original code in <code>{path2.name}</code></h5>
                            <div class="code-block"><pre>{call_2}</pre></div>
                        </div>
                    </div>
                </div>
                """
                html_parts.append(html_block)

            else:
                html_parts.append(f'<div class="refactoring-block"><h4>Refactoring Suggestion for Group {i+1}</h4><p>Could not parse both files for structural analysis.</p></div>')
    else:
        html_parts.append("<p>No clone groups found to suggest refactoring.</p>")
        
    html_parts.append('</section>') # end section

    # ... (Closing HTML tags) ...
    html_parts.append('</div></body></html>')

    # Write the file
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write('\n'.join(html_parts))

    print(f"Comprehensive report generated: {output_filename}")
    webbrowser.open(f"file://{Path(output_filename).resolve()}")


def visualize_clones_html(clone_groups, output_filename="clone_diff_report.html"):
    """
    Generates an HTML file with side-by-side diffs for clone pairs.
    """
    if not clone_groups:
        print("No clones to visualize.")
        return

    html_parts = [
        '<html><head><title>Clone Diff Report</title>',
        '<style>body { font-family: sans-serif; } table { border-collapse: collapse; }',
        'td { padding: 2px; } .diff_header { background-color: #e0e0e0; }',
        'h1, h2 { padding: 10px; background-color: #f0f0f0; border-radius: 5px; }',
        '</style></head><body><h1>Clone Detection Report</h1>'
    ]

    # Use HtmlDiff for a side-by-side comparison
    html_diff = difflib.HtmlDiff(tabsize=4, wrapcolumn=50)

    for i, group in enumerate(clone_groups):
        html_parts.append(f"<h2>Clone Group {i+1}</h2>")
        for path1, path2 in combinations(sorted(list(group)), 2):
            try:
                with open(path1, 'r', encoding='utf-8') as f1, open(path2, 'r', encoding='utf-8') as f2:
                    file1_lines = f1.readlines()
                    file2_lines = f2.readlines()

                # Generate the HTML table for the diff
                diff_table = html_diff.make_table(
                    file1_lines,
                    file2_lines,
                    fromdesc=str(path1.name),
                    todesc=str(path2.name)
                )
                html_parts.append(diff_table)

            except Exception as e:
                html_parts.append(f"<p>Could not generate diff for {path1.name} and {path2.name}: {e}</p>")

    html_parts.append('</body></html>')

    # Write the final HTML file
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write('\n'.join(html_parts))

    print(f"HTML report generated: {output_filename}")
    # Open the report in the default web browser
    webbrowser.open(f"file://{Path(output_filename).resolve()}")


def visualize_clone_graph(clone_pairs, output_filename="clone_graph.html"):
    """
    Generates an interactive HTML graph of clone relationships.
    Writes the generated HTML using UTF-8 to avoid encoding errors on Windows.
    """
    if not clone_pairs:
        print("No clone pairs to visualize in a graph.")
        return

    net = Network(notebook=True, directed=False, cdn_resources='in_line')
    
    # Use NetworkX to handle graph properties like connected components (groups)
    G = nx.Graph()
    for path1, path2, distance in clone_pairs:
        G.add_edge(str(path1.name), str(path2.name), weight=distance, title=f"Distance: {distance}")

    # Assign a group ID to each component for coloring
    for i, component in enumerate(nx.connected_components(G)):
        for node in component:
            if node in G:
                G.nodes[node]['group'] = i

    net.from_nx(G)
    net.show_buttons(filter_=['physics'])

    # Generate the HTML and write it explicitly with UTF-8 encoding to avoid
    # UnicodeEncodeError on platforms that use a limited default encoding.
    try:
        html = net.generate_html()
        with open(output_filename, 'w', encoding='utf-8') as f:
            f.write(html)
        print(f"Interactive graph saved to {output_filename}")
        webbrowser.open(f"file://{Path(output_filename).resolve()}")
    except Exception as e:
        print(f"Failed to generate or save interactive graph: {e}")


In [19]:
#root_dir = 'C:/Users/Falco/Documents/Università/EQS/terraform-examples/'  # Change as needed
root_dir = 'C:/Users/Falco/Documents/Università/EQS/Materiale/TerraDS'
#iac_files = find_iac_files(root_dir, limit=50)  # Limit to first 50 files for testing


In [20]:
# You can adjust the threshold. A lower value means stricter cloning.
# threshold=0 would find only structurally identical files.
# clone_pairs_with_distance = detect_clones_zss(iac_files, threshold=2)
# clone_pairs_with_distance = detect_clones_zss_parallel(iac_files, threshold=2)
clone_pairs_with_distance = detect_clones_smart(root_dir, limit=1000, threshold=5)

2025-12-04 13:17:11,801 - INFO - Found 1000 files. Starting parsing and bucketing...
2025-12-04 13:17:19,522 - INFO - Parsing complete. Skipped 0 files.
2025-12-04 13:17:19,523 - INFO - Created 418 distinct buckets based on structure.
2025-12-04 13:17:19,524 - INFO - Active buckets to process: 90
2025-12-04 13:17:19,524 - INFO - Estimated comparisons required: 596 (instead of classic N^2)
2025-12-04 13:18:08,167 - INFO - Detection complete. Found 356 clone pairs.


In [21]:

# We need to reconstruct the clone groups for the other functions
clone_groups_from_pairs = list(nx.connected_components(nx.Graph([(p1, p2) for p1, p2, _ in clone_pairs_with_distance])))

report_zss(clone_groups_from_pairs)



Clone group 1: 2 files
  - C:\Users\Falco\Documents\Università\EQS\Materiale\TerraDS\100137176\100137176\s3.tf
  - C:\Users\Falco\Documents\Università\EQS\Materiale\TerraDS\100201679\100201679\tf_remote_s3_bucket\s3.tf

Clone group 2: 2 files
  - C:\Users\Falco\Documents\Università\EQS\Materiale\TerraDS\100137176\100137176\iam_role.tf
  - C:\Users\Falco\Documents\Università\EQS\Materiale\TerraDS\100201679\100201679\tf_remote_s3_bucket\iam_role.tf

Clone group 3: 5 files
  - C:\Users\Falco\Documents\Università\EQS\Materiale\TerraDS\100425660\100425660\infrastructure\mgt\modules\13e9db568733e29a77283f4471baa261\nat\nat.tf
  - C:\Users\Falco\Documents\Università\EQS\Materiale\TerraDS\100425660\100425660\infrastructure\mgt\modules\58f53f37bccd55bbec80511f5fcec98d\nat.tf
  - C:\Users\Falco\Documents\Università\EQS\Materiale\TerraDS\100425660\100425660\infrastructure\mgt\modules\6d54864e409d259d1213453772f38d59\nat\nat.tf
  - C:\Users\Falco\Documents\Università\EQS\Materiale\TerraDS\1004256

In [22]:
#visualize_clones_diff(clone_groups_from_pairs)


In [23]:
#visualize_clones_html(clone_groups_from_pairs)


In [24]:
#visualize_clone_graph(clone_pairs_with_distance)

In [25]:
suggest_refactorings(clone_groups_from_pairs)


Refactoring Suggestions (Smart Mode - FIXED)

--- Clone Group 1 ---
Prototype Files: s3.tf <--> s3.tf
  Status: Identical files (deduplication candidate).

--- Clone Group 2 ---
Prototype Files: iam_role.tf <--> iam_role.tf
  Status: Identical files (deduplication candidate).

--- Clone Group 3 ---
Prototype Files: nat.tf <--> nat.tf
  Status: Identical files (deduplication candidate).

--- Clone Group 4 ---
Prototype Files: network.tf <--> network.tf
  Status: Identical files (deduplication candidate).

--- Clone Group 5 ---
Prototype Files: private_subnet.tf <--> private_subnet.tf
  Status: Identical files (deduplication candidate).

--- Clone Group 6 ---
Prototype Files: public_subnet.tf <--> public_subnet.tf
  Status: Identical files (deduplication candidate).

--- Clone Group 7 ---
Prototype Files: 030-security-group.tf <--> 030-security-group.tf
  Status: Identical files (deduplication candidate).

--- Clone Group 8 ---
Prototype Files: main.tf <--> main.tf
  Detected 1 parametr

In [26]:
# Generate the comprehensive report with all visualizations
generate_comprehensive_report(clone_pairs_with_distance, clone_groups_from_pairs)

Comprehensive report generated: clone_report.html
